# Tender corpus source-quality audit

Review date: 6 October 2026. This notebook presents retained executed SQL and results from two publisher-hash-verified SQLite snapshots. It does not require bundled databases, network access or a notebook server to inspect. Code cells are optional and deliberately unexecuted here; results below are copied from the retained query receipts, not fabricated notebook execution.

The snapshot adds original notice tables to the site's existing award-derived corpus. Listing rows, detail rows, candidate links, awards and payments are different grains. No supplier identities, political influence or payment paths are inferred from this audit.


In [ ]:
from pathlib import Path
import json, hashlib
ROOT = Path.cwd()
while not (ROOT / 'research/raw/tender-portal-audit/analysis.json').exists():
    if ROOT.parent == ROOT: raise FileNotFoundError('Run inside the repository')
    ROOT = ROOT.parent
AUDIT = ROOT / 'research/raw/tender-portal-audit'
read = lambda name: json.loads((AUDIT / name).read_text())
analysis, followup = read('analysis.json'), read('followup.json')
semantics, linkage = read('closing-field-semantics.json'), read('linkage-v2.json')
analysis['sourceHashes']


## Pinned original inputs

- `aoc_tenders.db`: SHA-256 `ec8ef7711a17b7cae9e0414c2403b119a0a31c4dec49ed7055b38ec0df5f7586`.
- `tenders_vps.db`: SHA-256 `b1994cfb6dd2d5da9ed1d9ac8d6bbc7083178f155e92a65628e87a38e4c64d01`.

Download receipts retain bytes, CRC, actual digest and public mirror. Databases are outside the repository at `/workspace/research-cache/tender-20260626/`. The award table is not newly independent evidence for the historical HF conversion.


## Grain and coverage

4,921,960 award listings and 3,952,191 notice listings are separate stages. Their detail tables are relational storage, not additional distinct tenders.

Retained query: `analysis.json#source_counts`

```sql
SELECT 'award_listings' AS table_name,count(*) AS n FROM src_a.aoc_tenders UNION ALL SELECT 'award_details',count(*) FROM src_a.aoc_details UNION ALL SELECT 'notice_listings',count(*) FROM src_t.tenders UNION ALL SELECT 'notice_details',count(*) FROM src_t.tender_details
```

Retained result:
```json
[
  {
    "table_name": "award_listings",
    "n": 4921960
  },
  {
    "table_name": "award_details",
    "n": 4540739
  },
  {
    "table_name": "notice_listings",
    "n": 3952191
  },
  {
    "table_name": "notice_details",
    "n": 3178485
  }
]
```


In [ ]:
read('analysis.json')["queries"]['source_counts']["rows"]


## Unsafe identity join

An ID-only join produces 2,730,656,872 candidate pairs. Generic reused IDs dominate. Cardinality is not a contract count.

Retained query: `analysis.json#join_id_only`

```sql
WITH n AS(SELECT tender_id,count(*) AS n FROM notices WHERE nullif(trim(tender_id),'') IS NOT NULL GROUP BY tender_id),a AS(SELECT tender_id,count(*) AS n FROM awards WHERE nullif(trim(tender_id),'') IS NOT NULL GROUP BY tender_id) SELECT count(*) AS shared_keys,sum(n.n::HUGEINT*a.n) AS candidate_pair_rows,sum(n.n) AS matched_notice_rows,sum(a.n) AS matched_award_rows,count(*) FILTER(WHERE n.n=1 AND a.n=1) AS one_to_one_keys,count(*) FILTER(WHERE n.n>1 AND a.n>1) AS many_to_many_keys,max(n.n::HUGEINT*a.n) AS max_pairs_one_key FROM n JOIN a ON n.tender_id=a.tender_id
```

Retained result:
```json
[
  {
    "shared_keys": 508296,
    "candidate_pair_rows": 2730656872,
    "matched_notice_rows": 1021326,
    "matched_award_rows": 1133339,
    "one_to_one_keys": 248988,
    "many_to_many_keys": 82932,
    "max_pairs_one_key": 2579823328
  }
]
```


In [ ]:
read('analysis.json')["queries"]['join_id_only']["rows"]


## Listing-only contextual join

Exact portal + buyer label + ID + nonblank listing reference yields 1,714 unique candidate pairs. This is not independent source confirmation.

Retained query: `analysis.json#join_portal_buyer_id_reference`

```sql
WITH n AS(SELECT portal,buyer,tender_id,reference,count(*) AS n FROM notices WHERE nullif(trim(portal),'') IS NOT NULL AND nullif(trim(buyer),'') IS NOT NULL AND nullif(trim(tender_id),'') IS NOT NULL AND nullif(trim(reference),'') IS NOT NULL GROUP BY portal,buyer,tender_id,reference),a AS(SELECT portal,buyer,tender_id,reference,count(*) AS n FROM awards WHERE nullif(trim(portal),'') IS NOT NULL AND nullif(trim(buyer),'') IS NOT NULL AND nullif(trim(tender_id),'') IS NOT NULL AND nullif(trim(reference),'') IS NOT NULL GROUP BY portal,buyer,tender_id,reference) SELECT count(*) AS shared_keys,sum(n.n::HUGEINT*a.n) AS candidate_pair_rows,sum(n.n) AS matched_notice_rows,sum(a.n) AS matched_award_rows,count(*) FILTER(WHERE n.n=1 AND a.n=1) AS one_to_one_keys,count(*) FILTER(WHERE n.n>1 AND a.n>1) AS many_to_many_keys,max(n.n::HUGEINT*a.n) AS max_pairs_one_key FROM n JOIN a ON n.portal=a.portal AND n.buyer=a.buyer AND n.tender_id=a.tender_id AND n.reference=a.reference
```

Retained result:
```json
[
  {
    "shared_keys": 2625,
    "candidate_pair_rows": 3877,
    "matched_notice_rows": 2625,
    "matched_award_rows": 3877,
    "one_to_one_keys": 1714,
    "many_to_many_keys": 0,
    "max_pairs_one_key": 12
  }
]
```


In [ ]:
read('analysis.json')["queries"]['join_portal_buyer_id_reference']["rows"]


## Declared detail-reference fallback

A separate fallback fills missing references from award details with the same within-stage internal_id. It yields 17,704 unique candidate pairs. Its population overlaps the first variant; do not add them.

Retained query: `linkage-v2.json#fallback_join_cardinality`

```sql
WITH n AS(SELECT portal,buyer,tender_id,trim(reference) AS reference_key,count(*) AS n FROM audit.notices WHERE nullif(trim(reference),'') IS NOT NULL AND nullif(trim(buyer),'') IS NOT NULL AND nullif(trim(tender_id),'') IS NOT NULL GROUP BY ALL),a AS(SELECT portal,buyer,tender_id,reference_key,count(*) AS n FROM award_keys WHERE reference_key IS NOT NULL AND nullif(trim(buyer),'') IS NOT NULL AND nullif(trim(tender_id),'') IS NOT NULL GROUP BY ALL) SELECT count(*) AS shared_keys,sum(n.n::HUGEINT*a.n) AS candidate_pairs,count(*) FILTER(WHERE n.n=1 AND a.n=1) AS one_to_one_keys,count(*) FILTER(WHERE n.n>1 AND a.n>1) AS many_to_many_keys FROM n JOIN a USING(portal,buyer,tender_id,reference_key)
```

Retained result:
```json
[
  {
    "shared_keys": 24809,
    "candidate_pairs": 34261,
    "one_to_one_keys": 17704,
    "many_to_many_keys": 3
  }
]
```


In [ ]:
read('linkage-v2.json')["queries"]['fallback_join_cardinality']["rows"]


## Closing-field semantics

1,709/1,714 award closing timestamps equal notice publication, and zero equal notice closing. All are central and dated 2013–2018; 1,632 belong to Neyveli Lignite. This is a selected subset, not a representative estimate for all awards.

Retained query: `closing-field-semantics.json#field_equality`

```sql
WITH nk AS(SELECT portal,buyer,tender_id,reference,count(*) AS n FROM notices WHERE nullif(trim(tender_id),'') IS NOT NULL AND nullif(trim(buyer),'') IS NOT NULL AND nullif(trim(reference),'') IS NOT NULL GROUP BY ALL),ak AS(SELECT portal,buyer,tender_id,reference,count(*) AS n FROM awards WHERE nullif(trim(tender_id),'') IS NOT NULL AND nullif(trim(buyer),'') IS NOT NULL AND nullif(trim(reference),'') IS NOT NULL GROUP BY ALL),matched AS(SELECT n.*,a.internal_id AS award_internal_id,a.aoc_at,a.closing_at AS award_closing,a.scraped_at AS award_scraped,a.bids_raw,a.value_raw,a.value_amount,a.document_url AS award_document_url,ak.n AS award_versions FROM notices n JOIN nk USING(portal,buyer,tender_id,reference) JOIN awards a USING(portal,buyer,tender_id,reference) JOIN ak USING(portal,buyer,tender_id,reference) WHERE nk.n=1 AND ak.n=1) SELECT count(*) AS pairs,count(*) FILTER(WHERE closing_at=award_closing) AS identical_closing_timestamp,count(*) FILTER(WHERE closing_at::DATE=award_closing::DATE) AS identical_closing_day,count(*) FILTER(WHERE award_closing=published_at) AS award_closing_equals_notice_publication,count(*) FILTER(WHERE award_closing::TIME=TIME '00:00:00') AS award_closing_midnight,min(published_at) AS earliest_notice,max(published_at) AS latest_notice FROM matched
```

Retained result:
```json
[
  {
    "pairs": 1714,
    "identical_closing_timestamp": 0,
    "identical_closing_day": 0,
    "award_closing_equals_notice_publication": 1709,
    "award_closing_midnight": 0,
    "earliest_notice": "2013-12-02 17:00:00",
    "latest_notice": "2018-12-31 12:35:00"
  }
]
```


In [ ]:
read('closing-field-semantics.json')["queries"]['field_equality']["rows"]


## Strict candidate scope

2,071 contextual-key rows meet declared ID, lexical and ordered notice/AOC-date criteria, but none confirms closing timestamp semantics. Two short-window/single-bid observations are leads only. Original timestamp-compatible candidate-pool.json remains empty.

Retained query: `linkage-v2.json#candidate_eligibility_by_closing_semantics`

```sql
SELECT count(*) AS exact_key_chronology_eligible,count(*) FILTER(WHERE a.closing_at=n.closing_at) AS closing_timestamp_confirmed,count(*) FILTER(WHERE a.closing_at=n.published_at) AS award_closing_is_notice_publication,count(*) FILTER(WHERE cast(a.bids_raw AS INTEGER)=1 AND date_diff('minute',n.published_at,n.closing_at)<72*60) AS short_window_single_bid FROM unique_pairs p JOIN audit.notices n ON n.internal_id=p.notice_internal_id JOIN audit.awards a ON a.internal_id=p.award_internal_id WHERE n.published_at>=TIMESTAMP '2021-06-16' AND n.published_at<=n.closing_at AND n.closing_at<=a.aoc_at AND a.aoc_at<=a.scraped_at AND regexp_full_match(n.tender_id,'[0-9]{4}_[A-Za-z][A-Za-z0-9]*_[0-9]+_[0-9]+') AND regexp_full_match(trim(a.bids_raw),'[0-9]+') AND try_cast(trim(a.bids_raw) AS INTEGER) BETWEEN 1 AND 1000 AND regexp_full_match(trim(a.value_raw),'[0-9]+(?:\.[0-9]+)?') AND a.value_amount BETWEEN 100000 AND 10000000000
```

Retained result:
```json
[
  {
    "exact_key_chronology_eligible": 2071,
    "closing_timestamp_confirmed": 0,
    "award_closing_is_notice_publication": 2071,
    "short_window_single_bid": 2
  }
]
```


In [ ]:
read('linkage-v2.json')["queries"]['candidate_eligibility_by_closing_semantics']["rows"]


## Scheduled windows and capture status

Of 46 state windows under the analyst screen of 72 hours, 36 were scheduled to close after capture. Only 10 occur among 2,721 past-scheduled-closing state notices. This is not a statutory minimum or failure rate.

Retained query: `followup.json#window_status_cohorts`

```sql
SELECT portal,status,closing_at>listing_scraped_at AS scheduled_to_close_after_scrape,count(*) AS listings,median(date_diff('minute',published_at,closing_at)/1440.0) AS median_scheduled_window_days,count(*) FILTER(WHERE date_diff('minute',published_at,closing_at)<72*60) AS scheduled_window_under_72h,count(*) FILTER(WHERE nullif(trim(corrigendum_url),'') IS NOT NULL) AS with_corrigendum_link FROM notices WHERE published_at<=closing_at AND published_at<=listing_scraped_at AND year(published_at)>=2011 GROUP BY ALL ORDER BY ALL
```

Retained result:
```json
[
  {
    "portal": "central",
    "status": "active",
    "scheduled_to_close_after_scrape": false,
    "listings": 3014,
    "median_scheduled_window_days": 15.987152777777776,
    "scheduled_window_under_72h": 34,
    "with_corrigendum_link": 377
  },
  {
    "portal": "central",
    "status": "active",
    "scheduled_to_close_after_scrape": true,
    "listings": 27735,
    "median_scheduled_window_days": 21.270833333333332,
    "scheduled_window_under_72h": 15,
    "with_corrigendum_link": 1903
  },
  {
    "portal": "central",
    "status": "archived",
    "scheduled_to_close_after_scrape": false,
    "listings": 3874114,
    "median_scheduled_window_days": 14.083333333333334,
    "scheduled_window_under_72h": 82068,
    "with_corrigendum_link": 550147
  },
  {
    "portal": "central",
    "status": "archived",
    "scheduled_to_close_after_scrape": true,
    "listings": 97,
    "median_scheduled_window_days": 84.93125,
    "scheduled_window_under_72h": 0,
    "with_corrigendum_link": 86
  },
  {
    "portal": "state",
    "status": "active",
    "scheduled_to_close_after_scrape": false,
    "listings": 2721,
    "median_scheduled_window_days": 11.0,
    "scheduled_window_under_72h": 10,
    "with_corrigendum_link": 441
  },
  {
    "portal": "state",
    "status": "active",
    "scheduled_to_close_after_scrape": true,
    "listings": 39104,
    "median_scheduled_window_days": 14.0,
    "scheduled_window_under_72h": 36,
    "with_corrigendum_link": 7119
  }
]
```


In [ ]:
read('followup.json')["queries"]['window_status_cohorts']["rows"]


## Scoped deduplication sensitivity

Adding portal and buyer to the old dedup key adds 385 groups; adding reference adds 1,351 relative to the historical key. Neither change identifies repeated physical work or duplicate payment.

Retained query: `analysis.json#scoped_dedup_counts`

```sql
SELECT (SELECT count(*) FROM (SELECT tender_id,bidder_norm,aoc_at FROM awards GROUP BY ALL)) AS legacy_rows,(SELECT count(*) FROM (SELECT portal,buyer,tender_id,bidder_norm,aoc_at FROM awards GROUP BY ALL)) AS scoped_rows,(SELECT count(*) FROM (SELECT portal,buyer,tender_id,reference,bidder_norm,aoc_at FROM awards GROUP BY ALL)) AS reference_scoped_rows
```

Retained result:
```json
[
  {
    "legacy_rows": 3385233,
    "scoped_rows": 3385618,
    "reference_scoped_rows": 3386584
  }
]
```


In [ ]:
read('analysis.json')["queries"]['scoped_dedup_counts']["rows"]


## Raw payload variants

Null-versus-present conflicts are included. Different raw values need version reconciliation; an arbitrary selected row is not necessarily authoritative.

Retained query: `followup.json#payload_variants_raw_null_inclusive`

```sql
WITH k AS(SELECT tender_id,bidder_norm,aoc_at,count(*) AS rows,count(DISTINCT struct_pack(v:=portal)) AS portals,count(DISTINCT struct_pack(v:=buyer)) AS buyers,count(DISTINCT struct_pack(v:=reference)) AS refs,count(DISTINCT struct_pack(v:=value_raw)) AS raw_values,count(DISTINCT struct_pack(v:=bids_raw)) AS raw_bids FROM awards GROUP BY ALL) SELECT count(*) AS old_key_groups,count(*) FILTER(WHERE buyers>1) AS buyer_variants_including_null,count(*) FILTER(WHERE refs>1) AS reference_variants_including_null,count(*) FILTER(WHERE raw_values>1) AS raw_amount_variants_including_null,count(*) FILTER(WHERE raw_bids>1) AS raw_bid_variants_including_null FROM k
```

Retained result:
```json
[
  {
    "old_key_groups": 3385233,
    "buyer_variants_including_null": 310,
    "reference_variants_including_null": 426,
    "raw_amount_variants_including_null": 1757,
    "raw_bid_variants_including_null": 928
  }
]
```


In [ ]:
read('followup.json')["queries"]['payload_variants_raw_null_inclusive']["rows"]


## Timezone-sensitive chronology

Stored timestamps have no asserted timezone. The >24-hour split is separate from same-day offsets. Future scheduled notice closings remain expected observations.

Retained query: `followup.json#timezone_aware_caution`

```sql
SELECT portal,count(*) FILTER(WHERE aoc_at>scraped_at AND aoc_at<=scraped_at+INTERVAL '24 hour') AS after_scrape_within_24h,count(*) FILTER(WHERE aoc_at>scraped_at+INTERVAL '24 hour') AS after_scrape_over_24h,count(*) FILTER(WHERE aoc_at>scraped_at+INTERVAL '7 day') AS after_scrape_over_7d FROM awards GROUP BY portal
```

Retained result:
```json
[
  {
    "portal": "central",
    "after_scrape_within_24h": 10,
    "after_scrape_over_24h": 191,
    "after_scrape_over_7d": 173
  },
  {
    "portal": "state",
    "after_scrape_within_24h": 8,
    "after_scrape_over_24h": 620,
    "after_scrape_over_7d": 554
  }
]
```


In [ ]:
read('followup.json')["queries"]['timezone_aware_caution']["rows"]


## Award lexical control

All nonblank award amount and bid-count lexemes fit the observed strict forms. The numerical-stripping defect found in notice EMD/fee strings is not evidence that HF award amounts were corrupted.

Retained query: `followup.json#award_numeric_lexical_quality`

```sql
SELECT count(*) AS rows,count(*) FILTER(WHERE nullif(trim(bids_raw),'') IS NOT NULL) AS nonblank_bid_counts,count(*) FILTER(WHERE regexp_full_match(trim(bids_raw),'[0-9]+')) AS strict_unsigned_integer_forms,count(*) FILTER(WHERE regexp_full_match(trim(bids_raw),'[0-9]+') AND try_cast(trim(bids_raw) AS INTEGER) BETWEEN 1 AND 1000) AS strict_eligible_bid_counts,count(*) FILTER(WHERE bids IS NOT NULL AND NOT coalesce(regexp_full_match(trim(bids_raw),'[0-9]+'),false)) AS numeric_extraction_without_integer_form,count(*) FILTER(WHERE value_amount IS NOT NULL) AS extracted_amounts,count(*) FILTER(WHERE regexp_full_match(trim(value_raw),'(?i)(?:(?:INR|Rs\.?|₹)\s*)?(?:[0-9]+|[0-9]{1,3}(?:,[0-9]{3})+|[0-9]{1,2}(?:,[0-9]{2})*,[0-9]{3})(?:\.[0-9]+)?')) AS strict_grouped_numeric_forms,count(*) FILTER(WHERE value_amount IS NOT NULL AND NOT coalesce(regexp_full_match(trim(value_raw),'(?i)(?:(?:INR|Rs\.?|₹)\s*)?(?:[0-9]+|[0-9]{1,3}(?:,[0-9]{3})+|[0-9]{1,2}(?:,[0-9]{2})*,[0-9]{3})(?:\.[0-9]+)?'),false)) AS numeric_extraction_without_money_form FROM awards
```

Retained result:
```json
[
  {
    "rows": 4921960,
    "nonblank_bid_counts": 4482989,
    "strict_unsigned_integer_forms": 4482989,
    "strict_eligible_bid_counts": 4412945,
    "numeric_extraction_without_integer_form": 0,
    "extracted_amounts": 4463564,
    "strict_grouped_numeric_forms": 4463564,
    "numeric_extraction_without_money_form": 0
  }
]
```


In [ ]:
read('followup.json')["queries"]['award_numeric_lexical_quality']["rows"]


## Strict monetary parsing and cache control

The independent raw lexeme review found 14 notice EMD and one fee string in scientific notation. The converter's stripping rule, also used by our initial exploratory macro, would turn `1e17` into `117`. Original SQLite strings are intact. `strict_parse.py` quarantines exponent notation and unsupported forms, preserves raw strings and accepts reviewed rupee prefixes and Indian/western grouping. EMD and fees are not estimated contract values or payments.

`derivation-manifest.json` fingerprints source hashes plus all four table-build statements and macros. Future changes fail closed instead of silently reusing parsed tables. Initial registration occurred at handoff after checking the retained executed build SQL. `followup.json.inputAnalysisSHA256` refers to the byte-identical analysis artifact before the later `derivationFingerprint` metadata key was appended; removing only that key and serializing with indent=2 reproduces it. No SQL or result rows changed.


In [ ]:
import sys
sys.path.insert(0, str(AUDIT))
from strict_parse import amount, bids
assert amount('1e17')['status'] == 'scientific-notation-quarantined'
assert amount('₹ 1,23,456.50')['value'] == '123456.50'
assert bids('1.5')['status'] == 'unrecognised-format'
original_analysis = dict(analysis)
original_analysis.pop('derivationFingerprint', None)
assert hashlib.sha256(json.dumps(original_analysis, indent=2).encode()).hexdigest() == followup['inputAnalysisSHA256']


## Optional read-only rerun

The lightweight CI tests use only committed receipts/results and Python's standard library. Full SQL reruns need the external databases and DuckDB 1.5.6 with its SQLite extension. Do not download or install them merely to inspect this notebook. The following cell reads an existing derived cache, never creates a database. For complete replay use the separate scripts and a fresh cache when its fingerprint changes. Original snapshots remain attached read-only by the producer.


In [ ]:
# Optional: requires the already prepared external cache and DuckDB environment.
# import duckdb
# CACHE = Path('/workspace/research-cache/tender-20260626')
# if not (CACHE / 'audit.duckdb').is_file(): raise FileNotFoundError('External cache unavailable')
# with duckdb.connect(str(CACHE / 'audit.duckdb'), read_only=True) as connection:
#     sql = semantics['queries']['field_equality']['sql']
#     print(connection.execute(sql).fetchall())


## Original-document follow-up and remaining gaps

A finite seven-case follow-up retains exact public IDs, references, raw date/value strings and access receipts. The supplied original portal pages did not expose usable case fields (invalid URL, unauthorized redirect, or HTTP 403); an HTTP 200 error page is not verification. Retained raw AOC detail labels identify separate Published Date and Contract Date fields, but fresh government-page confirmation remains unavailable. The IRCTC contracts may involve licence revenue: cash direction is unresolved and amounts are not described as government spending. The comparison case illustrates different bid counts within a similar buyer/category/year; it is not a randomized or statistically matched control.

Required next documents are the original notice and amendments, award/contract, scope/lot or asset IDs, bills, payment and completion records. No amount is promoted to a disbursement, no supplier string is resolved to a legal entity, and no political graph join is made by name.
